# XGBOOST REGRESSION

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
data_df = pd.read_csv("data_cleaned.csv")
print(f"Loaded {len(data_df)} rows, {len(data_df.columns)} columns")
print(f"Columns: {list(data_df.columns)}")
data_df.head()

Loaded 1328 rows, 9 columns
Columns: ['age', 'bmi', 'children', 'charges', 'gender_encoded', 'smoker_encoded', 'region_northwest', 'region_southeast', 'region_southwest']


,age,bmi,children,charges,gender_encoded,smoker_encoded,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,1,1,0.0,0.0,1.0
1,18,33.770,1,1725.55230,0,0,0.0,1.0,0.0
2,28,33.000,3,4449.46200,0,0,0.0,1.0,0.0
3,33,22.705,0,21984.47061,0,0,1.0,0.0,0.0
4,32,28.880,0,3866.85520,0,0,1.0,0.0,0.0


## 1. Data Splitting

In [2]:
from sklearn.model_selection import train_test_split

data_df['smoker_bmi'] = data_df['smoker_encoded'] * data_df['bmi']
data_df['age_squared'] = data_df['age'] ** 2

feature_cols = ['age', 'age_squared', 'bmi', 'children', 'gender_encoded', 'smoker_encoded',
                'region_northwest', 'region_southeast', 'region_southwest',
                'smoker_bmi']

X = data_df[feature_cols]
y = data_df['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

Training set: 1062 samples
Test set:     266 samples


## 2. Model Training (Tuned to Reduce Overfitting)

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

# Baseline model (for before/after comparison)
baseline_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    objective='reg:squarederror',
    eval_metric='rmse'
)
baseline_model.fit(X_train, y_train)
y_pred_train_baseline = baseline_model.predict(X_train)
y_pred_test_baseline = baseline_model.predict(X_test)

# Split train into train/validation for early stopping (no test leakage)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

base_model = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42
)

param_dist = {
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0.0, 0.1, 0.5, 1.0],
    'reg_lambda': [1.0, 2.0, 5.0, 10.0],
    'n_estimators': [300, 500, 800]
}

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=1,
    verbose=0
)
search.fit(X_tr, y_tr)

tuned_params = search.best_params_.copy()

# Monitoring model: validation set for early stopping diagnostics
monitoring_model = XGBRegressor(
    **tuned_params,
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    early_stopping_rounds=50
)
monitoring_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    verbose=False
)

evals_result = monitoring_model.evals_result()
tuned_n_estimators = int(monitoring_model.best_iteration + 1)

# Final model: retrain on full training data with tuned complexity
model = XGBRegressor(
    **{k: v for k, v in tuned_params.items() if k != 'n_estimators'},
    n_estimators=tuned_n_estimators,
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42
)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print('XGBoost tuning completed.')
print('Best params:', tuned_params)
print(f'Best boosting rounds from early stopping: {tuned_n_estimators}')

## 3. Model Evaluation (R2 + RMSE + MAE)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# Tuned model metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)

# Baseline metrics (before tuning)
r2_train_base = r2_score(y_train, y_pred_train_baseline)
r2_test_base = r2_score(y_test, y_pred_test_baseline)
rmse_train_base = np.sqrt(mean_squared_error(y_train, y_pred_train_baseline))
rmse_test_base = np.sqrt(mean_squared_error(y_test, y_pred_test_baseline))
mae_train_base = mean_absolute_error(y_train, y_pred_train_baseline)
mae_test_base = mean_absolute_error(y_test, y_pred_test_baseline)

metrics = pd.DataFrame({
    'Metric': ['R2', 'RMSE', 'MAE'],
    'Train': [r2_train, rmse_train, mae_train],
    'Test': [r2_test, rmse_test, mae_test]
})

comparison_before_after = pd.DataFrame([
    {
        'Model': 'Baseline',
        'R2_train': r2_train_base,
        'R2_test': r2_test_base,
        'RMSE_train': rmse_train_base,
        'RMSE_test': rmse_test_base,
        'MAE_train': mae_train_base,
        'MAE_test': mae_test_base,
    },
    {
        'Model': 'Tuned',
        'R2_train': r2_train,
        'R2_test': r2_test,
        'RMSE_train': rmse_train,
        'RMSE_test': rmse_test,
        'MAE_train': mae_train,
        'MAE_test': mae_test,
    }
])

print('Model Performance (recommended metrics):\n')
print(metrics.to_string(index=False, formatters={'Train': '{:,.4f}'.format, 'Test': '{:,.4f}'.format}))

print('\nBefore vs After Tuning:\n')
display(comparison_before_after.style.format({
    'R2_train': '{:.4f}', 'R2_test': '{:.4f}',
    'RMSE_train': '{:,.2f}', 'RMSE_test': '{:,.2f}',
    'MAE_train': '{:,.2f}', 'MAE_test': '{:,.2f}'
}))

## 3b. Cross-Validation (5-Fold, RMSE-Focused, Tuned Params)

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from xgboost import XGBRegressor
import pandas as pd

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=42,
    objective='reg:squarederror',
    eval_metric='rmse'
)

cv_scores = cross_validate(
    cv_model,
    X,
    y,
    cv=cv,
    scoring={
        'rmse': 'neg_root_mean_squared_error',
        'mae': 'neg_mean_absolute_error',
        'r2': 'r2'  # optional overview metric
    },
    return_train_score=False,
    n_jobs=1
)

cv_result = pd.DataFrame({
    'Fold': [f'Fold {i}' for i in range(1, 6)],
    'RMSE': -cv_scores['test_rmse'],
    'MAE': -cv_scores['test_mae'],
    'R2 (optional)': cv_scores['test_r2']
})

cv_summary = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'R2 (optional)'],
    'Mean (5-Fold CV)': [
        cv_result['RMSE'].mean(),
        cv_result['MAE'].mean(),
        cv_result['R2 (optional)'].mean()
    ],
    'Std Dev': [
        cv_result['RMSE'].std(ddof=1),
        cv_result['MAE'].std(ddof=1),
        cv_result['R2 (optional)'].std(ddof=1)
    ]
})

print('Per-fold CV scores (RMSE-focused):')
display(cv_result.style.format({'RMSE': '{:,.2f}', 'MAE': '{:,.2f}', 'R2 (optional)': '{:.4f}'}))

print('\nCV summary (mean ± std):')
display(cv_summary.style.format({'Mean (5-Fold CV)': '{:,.4f}', 'Std Dev': '{:,.4f}'}))

## 4. Feature Importance (XGBoost)

In [ ]:
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Feature Importance (XGBoost):\n")
for _, row in importance_df.iterrows():
    print(f"  {row['Feature']:25s} : {row['Importance']:>12.4f}")

plt.figure(figsize=(10, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='#2ecc71')
plt.xlabel('Importance Score')
plt.title('XGBoost Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Residual Analysis

In [ ]:
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test, y_pred_test, alpha=0.5, edgecolors='black', linewidths=0.5)
max_val = max(y_test.max(), y_pred_test.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Charges')
axes[0].set_ylabel('Predicted Charges')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

axes[1].scatter(y_pred_test, residuals, alpha=0.5, edgecolors='black', linewidths=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Charges')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Predicted')

axes[2].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[2].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residual Value')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

## 6. Training Monitoring (Learning Curve)

In [ ]:
# Learning curve from eval_set (RMSE per boosting iteration)
train_rmse_curve = evals_result['validation_0']['rmse']
test_rmse_curve = evals_result['validation_1']['rmse']

rounds = range(1, len(train_rmse_curve) + 1)

plt.figure(figsize=(10, 5))
plt.plot(rounds, train_rmse_curve, label='Train RMSE')
plt.plot(rounds, test_rmse_curve, label='Test RMSE')
plt.xlabel('Boosting Iteration')
plt.ylabel('RMSE')
plt.title('XGBoost Learning Curve (Train vs Test RMSE)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_round = int(np.argmin(test_rmse_curve)) + 1
best_test_rmse_curve = float(np.min(test_rmse_curve))
final_gap = float(test_rmse_curve[-1] - train_rmse_curve[-1])

print(f'Best test RMSE during training: {best_test_rmse_curve:,.4f} at round {best_round}')
print(f'Final train-test RMSE gap: {final_gap:,.4f}')
if final_gap > 500:
    print('Potential overfitting signal: train error much lower than test error.')
else:
    print('No strong overfitting signal from train-test RMSE gap.')

# XGBoost Regression Summary

## Features Used (10)

| Feature | Type | Description |
|---------|------|-------------|
| age | Original | Patient age |
| age_squared | Polynomial | `age^2` to capture non-linear age effect |
| bmi | Original | Body Mass Index |
| children | Original | Number of dependents |
| gender_encoded | Encoded | 0 = male, 1 = female |
| smoker_encoded | Encoded | 0 = no, 1 = yes |
| region_northwest | OneHot | Region dummy |
| region_southeast | OneHot | Region dummy |
| region_southwest | OneHot | Region dummy |
| smoker_bmi | Interaction | `smoker_encoded * bmi` |

## Evaluation Setup Used

- **Primary emphasis**: RMSE
- **Companion metrics**: MAE and R2
- **Training monitor**: `eval_set=[(X_train, y_train), (X_test, y_test)]` with `eval_metric='rmse'`
- **Validation stability**: 5-Fold cross-validation (RMSE + MAE, R2 optional)
- **Diagnostics**: residual plot + learning curve (train vs test RMSE)

## Notes

XGBoost does not require linearity or normal residual assumptions, so performance evaluation focuses on prediction errors and generalization stability.